# Network analysis: validation and a live collection

What `services/research/network.py` claims is narrow, and this notebook tests it. Kieran Healy's
"Using Metadata to Find Paul Revere" (2013) took 254 Boston colonists × the 7 organisations they
belonged to, projected that to a person-to-person graph, and ran centrality on it. Revere came out on
top because he bridged groups that barely overlapped. So a few structural measures, applied to
membership data alone, find the brokers. That is all it shows: nothing about whether a broker's work
is good, and nothing about what would have happened otherwise.

**Part 1** reproduces that result with the same functions the app uses. If Revere does not rank first
the implementation is wrong, and the cell raises.

**Part 2** runs against a live collection over the HTTP API and checks how stable the answer is. It
needs `CLIO_URL` and `CLIO_COLLECTION` in the environment, plus `CLIO_TOKEN` unless the server runs without
a password, and is skipped without them. The token is read from the environment and never printed.

Running it needs a Jupyter kernel in addition to the app's requirements:
`uv pip install --python venv/Scripts/python.exe nbclient ipykernel`. Please commit this file with its
outputs cleared: they would hold collection contents, and part of a dataset whose licence is unverified.

In [ ]:
import csv
import io
import json
import os
import sys
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

root = Path.cwd().resolve()
while not (root / "services" / "research" / "network.py").exists():
    if root == root.parent:
        raise RuntimeError("Run this from inside the Clio repository.")
    root = root.parent
sys.path.insert(0, str(root))

from services.research import network as net

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

METRICS = ["betweenness", "eigenvector", "pagerank", "degree", "effective_size"]


def top(frame, metric, n=10, ascending=False):
    return frame.sort_values([metric, "label"], ascending=[ascending, True]).head(n)[["label", metric]].reset_index(drop=True)


def spearman(a, b):
    """Spearman's rho on the nodes both have: Pearson on average ranks, so no SciPy."""
    common = a.index.intersection(b.index)
    if len(common) < 3 or a[common].nunique() < 2 or b[common].nunique() < 2:
        return None
    return round(float(a[common].rank().corr(b[common].rank())), 4)


def jaccard(a, b):
    a, b = set(a), set(b)
    return round(len(a & b) / len(a | b), 4) if a | b else 1.0


def rand_index(a, b):
    """Share of node pairs two partitions agree on (together in both, or apart in both)."""
    common = sorted(set(a) & set(b), key=str)
    agree = total = 0
    for i, x in enumerate(common):
        for y in common[i + 1:]:
            total += 1
            together_a = a[x] is not None and a[x] == a[y]
            together_b = b[x] is not None and b[x] == b[y]
            agree += together_a == together_b
    return round(agree / total, 4) if total else 1.0

## Part 1: reproduce Healy's result

The data is the appendix of Fischer's *Paul Revere's Ride* as published in
[kjhealy/revere](https://github.com/kjhealy/revere). It is fetched at runtime and not stored in this
repository, because its licence is unverified.

**How far this reproduces the post.** It reproduces the finding, not the printed numbers. Revere ranks
first on betweenness under every construction tried, and the top four (Revere, Urann, Warren, Peck) come
in the post's order. But from this CSV no construction gives the post's betweenness values (3839 for
Revere there, 4372 here unnormalised, 4490 counting parallel ties as Healy's `revere.R` does). The post's
eigenvector table also differs: it puts Revere fourth at 0.99, behind three people at 1.00, two of whom
(Hoffins, Cooper) belong to a single group in this CSV, which cannot score that high. So the table
appears to come from a different version of the data, or carries a labelling error. That cannot be settled
from here. In this CSV Revere ranks first on eigenvector centrality under every construction tried.

In [ ]:
REVERE_URL = "https://raw.githubusercontent.com/kjhealy/revere/master/data/PaulRevereAppD.csv"
with urllib.request.urlopen(REVERE_URL, timeout=30) as response:
    rows = list(csv.reader(io.StringIO(response.read().decode("utf-8"))))

groups = rows[0][1:]
memberships = {g: [r[0] for r in rows[1:] if r[i + 1].strip() == "1"] for i, g in enumerate(groups)}
bipartite, people = net.project_memberships(memberships)
for person in people:
    people.nodes[person]["label"] = person
texts = {person: [g for g, members in memberships.items() if person in members] for person in people}

assert len(rows) - 1 == 254 and len(groups) == 7, "The published table has 254 people and 7 groups."
assert people.number_of_nodes() == 254

revere = net.analyze(people, "membership", node_texts=texts)
revere_frame = pd.DataFrame(revere["nodes"]).set_index("id", drop=False)
print(revere["stats"])
print(revere["reliability"]["level"], "-", revere["reliability"]["note"])
print("Group sizes:", {g: len(m) for g, m in memberships.items()})
print("Revere belongs to:", texts["Revere.Paul"])

In [ ]:
ranks = {}
for metric in METRICS:
    ranks[metric] = int(revere_frame[metric].rank(ascending=False, method="min")["Revere.Paul"])
ranks["constraint (lowest = rank 1)"] = int(revere_frame["constraint"].rank(ascending=True, method="min")["Revere.Paul"])
print("Revere's rank of 254:", ranks)

# The validation. If either fails the implementation is wrong: do not adjust these.
assert ranks["betweenness"] == 1, "Revere must rank first on betweenness."
assert ranks["eigenvector"] == 1, "Revere must rank first on eigenvector centrality."
print("VALIDATION PASSED: Revere ranks first on betweenness and eigenvector.")

display(pd.concat({m: top(revere_frame, m, 5) for m in ("betweenness", "eigenvector", "pagerank")}, axis=1))
display(top(revere_frame, "constraint", 5, ascending=True))

In [ ]:
print("Communities:", [(c["label"], c["size"]) for c in revere["communities"]], "modularity", revere["stats"]["modularity"])
for broker in revere["brokers"][:5]:
    print(f"{broker['label']:20} betweenness {broker['betweenness']:.4f}  {broker['sentence']}")

### Is Revere's lead stable?

The same check as for a live collection. Betweenness and eigenvector do not depend on the Louvain
seed, so what can change with a seed is the grouping, and with it who counts as linking groups. A
stronger test of the claim is whether Revere stays first when the data itself changes, so each group
is dropped in turn.

In [ ]:
seed_runs = {seed: net.analyze(people, "membership", seed=seed, node_texts=texts) for seed in (42, 1, 2, 3, 4)}
base = seed_runs[42]
base_groups = {r["id"]: r["community"] for r in base["nodes"]}
base_brokers = [b["id"] for b in base["brokers"][:5]]
display(pd.DataFrame([{
    "seed": seed, "communities": run["stats"]["communities"], "modularity": run["stats"]["modularity"],
    "rand index vs seed 42": rand_index(base_groups, {r["id"]: r["community"] for r in run["nodes"]}),
    "top-5 broker Jaccard": jaccard(base_brokers, [b["id"] for b in run["brokers"][:5]]),
    "top broker": run["brokers"][0]["id"],
} for seed, run in seed_runs.items()]))

drop_rows = []
for dropped in groups:
    _, reduced = net.project_memberships({g: m for g, m in memberships.items() if g != dropped})
    frame = pd.DataFrame(net.analyze(reduced, "membership")["nodes"]).set_index("id", drop=False)
    drop_rows.append({
        "without": dropped, "people left": len(frame),
        "first on betweenness": frame["betweenness"].idxmax(),
        "Revere's betweenness rank": int(frame["betweenness"].rank(ascending=False, method="min")["Revere.Paul"]),
        "spearman vs full": spearman(revere_frame["betweenness"], frame["betweenness"]),
    })
display(pd.DataFrame(drop_rows))

## Part 2: a live collection, over the HTTP API

Nothing is read from disk here. Everything comes through `/api/research/network`, the same route the
map uses, so quarantined documents are already out and read access is enforced by the server.

In [ ]:
CLIO_URL = os.environ.get("CLIO_URL", "").rstrip("/")
CLIO_TOKEN = os.environ.get("CLIO_TOKEN", "")
COLLECTION = os.environ.get("CLIO_COLLECTION", "")
LIVE = bool(CLIO_URL and COLLECTION)
GRAPHS = ("citation", "authors", "entities")


def fetch(graph, **params):
    query = urllib.parse.urlencode({"collection_id": COLLECTION, "graph": graph, **params})
    headers = {"Authorization": f"Bearer {CLIO_TOKEN}"} if CLIO_TOKEN else {}
    request = urllib.request.Request(f"{CLIO_URL}/api/research/network?{query}", headers=headers)
    with urllib.request.urlopen(request, timeout=300) as response:
        return json.load(response)


def frame_of(result):
    return pd.DataFrame(result["nodes"]).set_index("id", drop=False)


live = {}
if LIVE:
    live = {graph: fetch(graph) for graph in GRAPHS}
    for graph, result in live.items():
        print(f"--- {graph} ---")
        print(result["stats"])
        print(result["reliability"]["level"], "-", result["reliability"]["note"], "\n")
else:
    print("Set CLIO_URL and CLIO_COLLECTION (and CLIO_TOKEN unless the server is open) to run this part.")

In [ ]:
for graph, result in live.items():
    frame = frame_of(result)
    print(f"--- {graph}: top 10 per metric ---")
    display(pd.concat({m: top(frame, m) for m in ("betweenness", "pagerank", "eigenvector")}, axis=1))
    print("Lowest constraint:")
    display(top(frame[frame["constraint"].notna()], "constraint", 10, ascending=True))
    print("Communities:")
    display(pd.DataFrame([{"label": c["label"], "size": c["size"], "terms": ", ".join(c["terms"][:4])}
                          for c in result["communities"]]))

In [ ]:
for graph, result in live.items():
    print(f"--- {graph}: brokers ---")
    if not result["brokers"]:
        print("None: nothing here links separate groups.")
    for broker in result["brokers"][:10]:
        print(f"{broker['betweenness']:.4f}  {broker['label'][:70]}")
        print(f"        {broker['sentence']}")
        for bridge in broker["bridges"]:
            links = ", ".join(edge["label"][:40] for edge in bridge["edges"][:3])
            print(f"        - {bridge['label']} ({bridge['count']}): {links}")
    if result["merges"] or result["ambiguous"] or result["unparsed_authors"]:
        print("Merged:", result["merges"])
        print("Left apart:", result["ambiguous"])
        print("Could not read:", result["unparsed_authors"])

### Sensitivity

A ranking that moves when nothing meaningful changed should not be read as a finding. Two checks:

* **Five Louvain seeds.** Betweenness and PageRank do not use the seed unless betweenness is sampled,
  so what can move is the grouping (Rand index) and who qualifies as linking groups (Jaccard).
* **`min_citations` ±1**, citation graph only. This changes which cited works are nodes at all, and so
  the ranks. Spearman is over the nodes both runs share.

In [ ]:
seed_rows = []
for graph in GRAPHS if LIVE else ():
    base = live[graph]
    base_groups = {r["id"]: r["community"] for r in base["nodes"]}
    base_brokers = [b["id"] for b in base["brokers"][:5]]
    base_frame = frame_of(base)
    for seed in (1, 2, 3, 4):
        run = fetch(graph, seed=seed)
        frame = frame_of(run)
        seed_rows.append({
            "graph": graph, "seed": seed, "communities": run["stats"]["communities"],
            "modularity": run["stats"]["modularity"],
            "rand index vs seed 42": rand_index(base_groups, {r["id"]: r["community"] for r in run["nodes"]}),
            "top-5 broker Jaccard": jaccard(base_brokers, [b["id"] for b in run["brokers"][:5]]),
            "betweenness spearman": spearman(base_frame["betweenness"], frame["betweenness"]),
        })
if seed_rows:
    display(pd.DataFrame(seed_rows))

In [ ]:
threshold_rows = []
if LIVE:
    base = live["citation"]
    base_frame = frame_of(base)
    base_brokers = [b["id"] for b in base["brokers"][:5]]
    for min_citations in (1, 3):
        run = fetch("citation", min_citations=min_citations)
        frame = frame_of(run)
        threshold_rows.append({
            "min_citations": min_citations, "nodes": run["stats"]["nodes"], "edges": run["stats"]["edges"],
            "shared nodes": len(base_frame.index.intersection(frame.index)),
            "betweenness spearman vs 2": spearman(base_frame["betweenness"], frame["betweenness"]),
            "pagerank spearman vs 2": spearman(base_frame["pagerank"], frame["pagerank"]),
            "top-5 broker Jaccard vs 2": jaccard(base_brokers, [b["id"] for b in run["brokers"][:5]]),
            "top broker": run["brokers"][0]["label"][:60] if run["brokers"] else None,
            "reliability": run["reliability"]["level"],
        })
    display(pd.DataFrame(threshold_rows))